# Solutions · 09 · Leaching and washing

Worked solutions to the exercises of [notebook 09](../notebooks/09_leaching_and_washing.ipynb). Try each exercise
yourself before reading its solution - the learning happens in the attempt. Solutions are one way to
answer each question; other correct approaches exist.

In [1]:
try:                      # on Google Colab (or anywhere engsep is missing): install it from GitHub
    import engsep
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ktwyw/engsep"], check=True)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from engsep import extraction as ex
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 90, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

import inspect
from IPython.display import Code

def show_source(obj):
    """Display the source code of a library function or class."""
    return Code(inspect.getsource(obj), language="python")

def heading(text):
    print(f"\n{text}\n" + "=" * len(text))

## Exercise 1

Sugar beet cossettes (17 % sugar, retaining 1.2 kg solution per kg dry pulp) are leached countercurrently with water to 99 % recovery. Find the stages for water-to-pulp ratios of 1.5, 2 and 3, and the sugar concentration of the juice. Which ratio gives the juice needing least evaporation?

In [2]:
sugar, retained = 0.17, 1.2                                          # kg sugar per kg cossettes (wet), kg solution per kg dry pulp
pulp = 1 - sugar                                                     # dry pulp per kg cossettes, treating the rest as insoluble... (approximation)
for ratio in (1.5, 2.0, 3.0):
    W = ratio / retained
    N = ex.leaching_stages(0.99, W)
    juice = ratio + sugar * 0.99                                       # water plus recovered sugar per kg cossettes
    conc = sugar * 0.99 / juice
    print(f"water/pulp = {ratio}: W = {W:.2f}, {N:.1f} stages for 99 % recovery; juice {conc:.1%} sugar -> {(1/conc - 1/0.65):.2f} kg water to evaporate per kg sugar for a 65 % syrup")

water/pulp = 1.5: W = 1.25, 13.6 stages for 99 % recovery; juice 10.1% sugar -> 8.37 kg water to evaporate per kg sugar for a 65 % syrup
water/pulp = 2.0: W = 1.67, 7.3 stages for 99 % recovery; juice 7.8% sugar -> 11.35 kg water to evaporate per kg sugar for a 65 % syrup
water/pulp = 3.0: W = 2.50, 4.5 stages for 99 % recovery; juice 5.3% sugar -> 17.29 kg water to evaporate per kg sugar for a 65 % syrup


More water means fewer stages but a thinner juice, and every kilogram of extra water must be boiled off in the
evaporators - typically the dominant energy cost of a sugar factory. The lowest water ratio that keeps the
stage count practical (here 1.5-2) minimises evaporation: leaching and evaporation are designed together.

## Exercise 2

Add a solute loss in the underflow moisture after pressing (the pulp is pressed to 0.6 kg/kg after the last stage): how does the recovery change, and how many extra stages compensate?

In [3]:
for pressed in (1.2, 0.6):
    base = ex.leaching_countercurrent(8, 2.0, 1.2, 100.0)
    loss = base["solute_lost"] * pressed / 1.2                          # the underflow solution carries the loss; pressing removes part of it back into the juice
    rec = 1 - loss / 100.0
    print(f"underflow moisture {pressed} kg/kg after the last stage: recovery {rec:.4f}")
N_extra = ex.leaching_stages(1 - ex.leaching_countercurrent(8, 2.0, 1.2, 100.0)["solute_lost"] * 0.6 / 1.2 / 100.0, 2.0 / 1.2) - 8
print(f"pressing the pulp from 1.2 to 0.6 kg/kg halves the loss, worth about {abs(N_extra):.1f} stages")

underflow moisture 1.2 kg/kg after the last stage: recovery 0.9932
underflow moisture 0.6 kg/kg after the last stage: recovery 0.9966
pressing the pulp from 1.2 to 0.6 kg/kg halves the loss, worth about 1.3 stages


The solute lost is what leaves in the last underflow's solution, so pressing the pulp to half its moisture
halves the loss - equivalent to about one extra stage. Presses are cheaper than stages, which is why every
diffuser has one.

## Exercise 3

**Implement it yourself:** simulate a 4-stage countercurrent leaching train by iterating the stage balances until the concentrations converge (constant underflow), and confirm the closed-form recovery.

In [4]:
N, W = 4, 2.0
c = np.full(N, 0.1)
for _ in range(1000):
    c_new = c.copy()
    for n in range(N):
        c_in_under = 1.0 if n == 0 else c[n - 1]                         # solute in the underflow from the previous stage (feed carries 1 unit in 1 unit of solution)
        c_in_over = 0.0 if n == N - 1 else c[n + 1]
        c_new[n] = (c_in_under + W * c_in_over) / (1 + W)                # ideal stage: both streams leave at c_n
    if np.max(np.abs(c_new - c)) < 1e-14:
        break
    c = c_new
print(f"iterated: recovery = {1 - c[-1]:.6f}; closed form: {ex.leaching_countercurrent(N, W, 1.0, 1.0)['recovery']:.6f}")

iterated: recovery = 0.967742; closed form: 0.967742


With constant underflow every stage is a linear mixing rule, and iterating the four of them reproduces the
Kremser closed form.